# 01 — Validação e preparação do HMDA 2024

No Colab, coloque o CSV nacional extraído no Google Drive em **Meu Drive/hmda-credit-fairness/data/raw/2024_public_lar_csv.csv**. Se preferir, altere CSV_PATH para apontar para o arquivo disponível. Execute as células em ordem. O Parquet e as tabelas de auditoria ficam em data/processed/valid_18. No Colab, coloque também scripts/contar_linhas_completas.py na pasta do projeto no Drive.

A leitura usa blocos e seleciona as colunas necessárias. Este notebook prepara uma única base com os 18 campos válidos. O notebook 02 faz a divisão estratificada em 70% para treinamento, 20% para validação e 10% para teste. A fonte e os códigos estão no [dicionário público do LAR](https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields).

In [5]:
from pathlib import Path
from collections import Counter
import sys

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

try:
    from google.colab import drive
except ImportError:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    CSV_PATH = ROOT / 'data/raw/2024_public_lar_csv/2024_public_lar_csv.csv'
else:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/hmda-credit-fairness')
    CSV_PATH = ROOT / 'data/raw/2024_public_lar_csv.csv'
OUTPUT = ROOT / 'data/processed/valid_18'
CHUNK_SIZE = 100_000

if not CSV_PATH.is_file():
    raise FileNotFoundError(f'Envie o CSV extraído ou ajuste CSV_PATH: {CSV_PATH}')

validation_script = ROOT / 'scripts/contar_linhas_completas.py'
if not validation_script.is_file():
    raise FileNotFoundError(f'Coloque o script compartilhado em {validation_script}')
sys.path.insert(0, str(validation_script.parent))
from contar_linhas_completas import COLUMNS, valid_fields

OUTPUT.mkdir(parents=True, exist_ok=True)

## Recorte e tipos

Incluímos compras de residência principal com decisão de aprovação ou recusa. O alvo vale 1 para os códigos 1 e 2 e 0 para o código 3. Idade e razão dívida/renda ficam como categorias, pois o LAR público contém faixas. Raça e etnia usam as categorias agregadas publicadas pelo HMDA, que podem refletir informações do solicitante e do co-solicitante. Depois dos filtros, aplicamos a auditoria dos 18 campos. Apenas pedidos com todas as colunas válidas são gravados. Valores inválidos ou indisponíveis, como idade 8888, ficam fora do Parquet.

In [6]:
BASE_NUM = ['income', 'loan_amount', 'loan_term', 'property_value']
BASE_CAT = ['lei', 'activity_year', 'applicant_age', 'debt_to_income_ratio',
            'loan_type', 'lien_status', 'construction_method', 'total_units']
DEMOGRAPHIC = ['derived_race', 'derived_ethnicity']
TRACT_INCOME = 'tract_to_msa_income_percentage'
ELIGIBILITY = ['loan_purpose', 'occupancy_type', 'action_taken']
FEATURES = BASE_NUM + BASE_CAT + DEMOGRAPHIC + [TRACT_INCOME]
INPUT_COLUMNS = FEATURES + ELIGIBILITY
assert len(INPUT_COLUMNS) == 18 and set(INPUT_COLUMNS) == set(COLUMNS)

header = pd.read_csv(CSV_PATH, nrows=0).columns
missing_columns = set(INPUT_COLUMNS) - set(header)
if missing_columns:
    raise ValueError(f'Colunas obrigatórias ausentes: {sorted(missing_columns)}')

schema = pa.schema(
    [(name, pa.float64()) for name in BASE_NUM]
    + [(name, pa.string()) for name in BASE_CAT + DEMOGRAPHIC]
    + [(TRACT_INCOME, pa.float64()), ('approved', pa.int8())],
    metadata={b'hmda_validated_fields': b'18'},
)

In [7]:
counts = Counter({key: 0 for key in ['lidos', 'hmda_2024', 'compra',
                                     'residencia_principal', 'decisao_elegivel',
                                     'descartados_validacao', 'validos_18_campos']})
actions = Counter()
target_counts = Counter()
invalid_counts = Counter({name: 0 for name in INPUT_COLUMNS})
categories = {name: Counter() for name in DEMOGRAPHIC}
parquet_path = OUTPUT / 'hmda_2024_purchase_valid18.parquet'
partial = parquet_path.with_suffix('.parquet.part')

with pq.ParquetWriter(partial, schema, compression='zstd') as writer:
    chunks = pd.read_csv(
        CSV_PATH, usecols=INPUT_COLUMNS, dtype='string', chunksize=CHUNK_SIZE,
        keep_default_na=False,
    )
    for chunk in chunks:
        counts['lidos'] += len(chunk)
        year = chunk.loc[chunk['activity_year'].eq('2024')]
        counts['hmda_2024'] += len(year)
        purchase = year.loc[year['loan_purpose'].eq('1')]
        counts['compra'] += len(purchase)
        primary = purchase.loc[purchase['occupancy_type'].eq('1')]
        counts['residencia_principal'] += len(primary)
        actions.update(primary['action_taken'].fillna('Ausente').value_counts().to_dict())

        eligible = primary.loc[primary['action_taken'].isin(['1', '2', '3'])]
        valid = valid_fields(eligible)
        accepted = valid.all(axis=1)
        counts['decisao_elegivel'] += len(eligible)
        counts['validos_18_campos'] += int(accepted.sum())
        counts['descartados_validacao'] += int((~accepted).sum())
        invalid_counts.update((~valid).sum().to_dict())
        if not accepted.any():
            continue

        selected = eligible.loc[accepted, FEATURES].copy()
        for name in BASE_NUM + [TRACT_INCOME]:
            selected[name] = pd.to_numeric(selected[name], errors='raise')
        selected['approved'] = eligible.loc[accepted, 'action_taken'].isin(['1', '2']).astype('int8')
        target_counts.update(selected['approved'].value_counts().to_dict())
        for name in DEMOGRAPHIC:
            categories[name].update(selected[name].value_counts().to_dict())

        writer.write_table(pa.Table.from_pandas(selected, schema=schema, preserve_index=False))

if set(target_counts) != {0, 1}:
    raise ValueError('O recorte precisa conter solicitações aprovadas e recusadas.')

with pq.ParquetFile(partial) as written:
    if written.metadata.num_rows != counts['validos_18_campos'] or sum(target_counts.values()) != counts['validos_18_campos']:
        raise ValueError('A quantidade gravada não corresponde à quantidade validada.')
assert counts['validos_18_campos'] + counts['descartados_validacao'] == counts['decisao_elegivel']
partial.replace(parquet_path)
print(f'Parquet pronto: {parquet_path}')

Parquet pronto: c:\Users\cg3034593\Documents\artigo\data\processed\valid_18\hmda_2024_purchase_valid18.parquet


In [8]:
def save_counts(values, label, filename, denominator=None):
    result = pd.Series(dict(values), dtype='int64', name='n').rename_axis(label).reset_index()
    if denominator:
        result['proporcao'] = result['n'] / denominator
    result.to_csv(OUTPUT / filename, index=False)
    display(result)

save_counts(counts, 'etapa', 'validacao_etapas.csv')
save_counts(actions, 'action_taken', 'validacao_acoes.csv')
save_counts(target_counts, 'approved', 'validacao_alvo.csv', counts['validos_18_campos'])
save_counts(invalid_counts, 'campo', 'validacao_invalidos_por_campo.csv', counts['decisao_elegivel'])
for name, values in categories.items():
    save_counts(values, name, f'validacao_{name}.csv', counts['validos_18_campos'])

,etapa,n
0,lidos,12229298
1,hmda_2024,12229298
2,compra,6553105
3,residencia_principal,5864623
4,decisao_elegivel,3794116
5,descartados_validacao,883474
6,validos_18_campos,2910642


,action_taken,n
0,1,3096020
1,6,1004583
2,3,557514
3,4,705753
4,2,140582
5,5,171112
6,8,144092
7,7,44967


,approved,n,proporcao
0,1,2570870,0.883266
1,0,339772,0.116734


,campo,n,proporcao
0,income,28908,0.007619
1,loan_amount,0,0.000000
2,loan_term,88533,0.023334
3,property_value,140683,0.037079
4,lei,0,0.000000
5,activity_year,0,0.000000
6,applicant_age,2118,0.000558
7,debt_to_income_ratio,118947,0.031350
8,loan_type,0,0.000000
9,lien_status,0,0.000000


,derived_race,n,proporcao
0,White,2228446,0.765620
1,Black or African American,301123,0.103456
2,Asian,243768,0.083751
3,Joint,96728,0.033233
4,American Indian or Alaska Native,25102,0.008624
5,2 or more minority races,8832,0.003034
6,Native Hawaiian or Other Pacific Islander,6643,0.002282


,derived_ethnicity,n,proporcao
0,Not Hispanic or Latino,2369203,0.813980
1,Hispanic or Latino,434933,0.149429
2,Joint,106506,0.036592
